# Transformer Hyperparameter Tuning

This notebook runs a hyperparameter tuning with the fairness rule below. 

Main fairness rule:
- fixed compact Transformer architectures
- same small grid style as CNN/LSTM/MLP
- model selection on clean validation score only
- 3-seed confirmation after choosing the best candidate

This makes the Transformer comparison safer. The goal is not to find the absolute best Transformer; the goal is to compare a reasonable Transformer under the same experimental discipline as the other families.

Import policy: this notebook imports the original project modules under `src/models/` and does not depend on any alternate file suffixes or renamed class aliases.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'. Run this notebook from inside the project repo.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [2]:
import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Keep deterministic behavior when possible.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [3]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.training import TrainConfig, train_from_tensors

from src.models.standard_transformer import (
    TransformerConfig,
    JointTransformerModel,
    MultiTaskTransformerModel,
    CoordinateTransformerModel,
)
from src.models.set_transformer import (
    SetTransformerConfig,
    JointSetTransformerModel,
    MultiTaskSetTransformerModel,
    CoordinateSetTransformerModel,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [4]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

in_dim = bundle.X_train.shape[1]

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


## Fixed architecture choices


In [5]:
# Fixed architecture choices for main fair comparison.
# These are intentionally compact and are not selected by Optuna architecture search.
STANDARD_ARCH = dict(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
)

SET_ARCH = dict(
    d_model=128,
    num_heads=4,
    num_sab_layers=2,
    dim_feedforward=256,
    num_seed_vectors=1,
)

STANDARD_ARCH, SET_ARCH


({'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256},
 {'d_model': 128,
  'num_heads': 4,
  'num_sab_layers': 2,
  'dim_feedforward': 256,
  'num_seed_vectors': 1})

## Shared helpers


In [6]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_cfg(
    *,
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=None,
    run_name=None,
):
    return TrainConfig(
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        val_batch_size=val_batch_size,
        max_epochs=max_epochs,
        patience=patience,
        print_every=print_every,
        grad_clip_norm=grad_clip_norm,
        run_name=run_name,
    )

def result_row(name, result):
    row = {"model": name, "best_epoch": result.best_epoch}
    row.update(result.best_metrics)
    return row

def run_trial(model, y_train, y_val, cfg, *, family, task, seed=42):
    set_seed(seed)
    result = train_from_tensors(
        model=model,
        X_train=bundle.X_train,
        y_train=y_train,
        X_val=bundle.X_val,
        y_val=y_val,
        device=device,
        cfg=cfg,
    )
    row = {
        "family": family,
        "task": task,
        "run_name": cfg.run_name,
        "seed": seed,
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "batch_size": cfg.batch_size,
        "val_batch_size": cfg.val_batch_size,
        "max_epochs": cfg.max_epochs,
        "patience": cfg.patience,
        "print_every": cfg.print_every,
        "grad_clip_norm": cfg.grad_clip_norm,
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row

def task_data(task_name):
    if task_name == "joint":
        return joint_y_train, joint_y_val
    if task_name == "multitask":
        return mt_y_train, mt_y_val
    if task_name == "coordinate":
        return coord_y_train, coord_y_val
    raise ValueError(task_name)

def build_model(family, task_name, *, dropout=0.10):
    if family == "standard":
        backbone_cfg = TransformerConfig(
            d_model=STANDARD_ARCH["d_model"],
            nhead=STANDARD_ARCH["nhead"],
            num_layers=STANDARD_ARCH["num_layers"],
            dim_feedforward=STANDARD_ARCH["dim_feedforward"],
            dropout=dropout,
        )
        if task_name == "joint":
            return JointTransformerModel(in_dim=in_dim, backbone_cfg=backbone_cfg)
        if task_name == "multitask":
            return MultiTaskTransformerModel(in_dim=in_dim, backbone_cfg=backbone_cfg)
        if task_name == "coordinate":
            return CoordinateTransformerModel(
                in_dim=in_dim,
                coordinate_std=bundle.coordinate_std,
                backbone_cfg=backbone_cfg,
            )

    if family == "set":
        backbone_cfg = SetTransformerConfig(
            d_model=SET_ARCH["d_model"],
            num_heads=SET_ARCH["num_heads"],
            num_sab_layers=SET_ARCH["num_sab_layers"],
            dim_feedforward=SET_ARCH["dim_feedforward"],
            dropout=dropout,
            num_seed_vectors=SET_ARCH["num_seed_vectors"],
        )
        if task_name == "joint":
            return JointSetTransformerModel(in_dim=in_dim, backbone_cfg=backbone_cfg)
        if task_name == "multitask":
            return MultiTaskSetTransformerModel(in_dim=in_dim, backbone_cfg=backbone_cfg)
        if task_name == "coordinate":
            return CoordinateSetTransformerModel(
                in_dim=in_dim,
                coordinate_std=bundle.coordinate_std,
                backbone_cfg=backbone_cfg,
            )

    raise ValueError((family, task_name))


## Parameter-count sanity check


In [7]:
param_rows = []

for family in ["standard", "set"]:
    for task_name in ["joint", "multitask", "coordinate"]:
        model = build_model(family, task_name, dropout=0.10)
        param_rows.append({
            "family": family,
            "task": task_name,
            "model": f"{family}_transformer_{task_name}",
            "params": count_trainable_params(model),
        })

param_df = pd.DataFrame(param_rows)
param_df["params_millions"] = param_df["params"] / 1_000_000
param_df


,family,task,model,params,params_millions
0,standard,joint,standard_transformer_joint,350861,0.350861
1,standard,multitask,standard_transformer_multitask,366728,0.366728
2,standard,coordinate,standard_transformer_coordinate,349442,0.349442
3,set,joint,set_transformer_joint,483213,0.483213
4,set,multitask,set_transformer_multitask,499080,0.499080
5,set,coordinate,set_transformer_coordinate,481794,0.481794


## Tuning grids

This is the final bounded Transformer tuning protocol for the main fair comparison.

Rules:
- fixed compact Standard Transformer architecture
- fixed compact Set Transformer architecture
- no Optuna
- no width/depth/head/seed-vector architecture search
- dropout fixed at 0.10
- five optimizer/schedule candidates per task and family
- selection on clean validation score only
- 3-seed confirmation after selection

This keeps Transformer tuning close to the CNN/LSTM/MLP tuning discipline.

In [8]:
BASELINE_CFG = dict(
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=1.0,
    dropout=0.10,
)

# Bounded 5-candidate grid.
# Architecture is fixed; dropout is fixed at 0.10.
# We only vary optimizer/schedule settings, matching the CNN/MLP tuning discipline.
def transformer_specs(prefix: str):
    return [
        dict(run_name=f"{prefix}_base_2e3_wd1e4",        lr=2e-3, weight_decay=1e-4, max_epochs=30, patience=5,  grad_clip_norm=1.0, dropout=0.10),
        dict(run_name=f"{prefix}_stable_1e3_wd1e4",     lr=1e-3, weight_decay=1e-4, max_epochs=50, patience=10, grad_clip_norm=1.0, dropout=0.10),
        dict(run_name=f"{prefix}_reg_1e3_wd5e4",        lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10, grad_clip_norm=1.0, dropout=0.10),
        dict(run_name=f"{prefix}_low_5e4_wd1e4",        lr=5e-4, weight_decay=1e-4, max_epochs=60, patience=12, grad_clip_norm=1.0, dropout=0.10),
        dict(run_name=f"{prefix}_lowreg_5e4_wd5e4",     lr=5e-4, weight_decay=5e-4, max_epochs=80, patience=15, grad_clip_norm=1.0, dropout=0.10),
    ]

TRANSFORMER_GRIDS = {
    "joint": [
        {**spec, "run_name": spec["run_name"].replace("xf", "{family}_joint")}
        for spec in transformer_specs("xf")
    ],
    "multitask": [
        {**spec, "run_name": spec["run_name"].replace("xf", "{family}_mt")}
        for spec in transformer_specs("xf")
    ],
    "coordinate": [
        {**spec, "run_name": spec["run_name"].replace("xf", "{family}_coord")}
        for spec in transformer_specs("xf")
    ],
}

BASELINE_CFG, TRANSFORMER_GRIDS

({'lr': 0.002,
  'weight_decay': 0.0001,
  'batch_size': 256,
  'val_batch_size': 512,
  'max_epochs': 30,
  'patience': 5,
  'print_every': 5,
  'grad_clip_norm': 1.0,
  'dropout': 0.1},
 {'joint': [{'run_name': '{family}_joint_base_2e3_wd1e4',
    'lr': 0.002,
    'weight_decay': 0.0001,
    'max_epochs': 30,
    'patience': 5,
    'grad_clip_norm': 1.0,
    'dropout': 0.1},
   {'run_name': '{family}_joint_stable_1e3_wd1e4',
    'lr': 0.001,
    'weight_decay': 0.0001,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': 1.0,
    'dropout': 0.1},
   {'run_name': '{family}_joint_reg_1e3_wd5e4',
    'lr': 0.001,
    'weight_decay': 0.0005,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': 1.0,
    'dropout': 0.1},
   {'run_name': '{family}_joint_low_5e4_wd1e4',
    'lr': 0.0005,
    'weight_decay': 0.0001,
    'max_epochs': 60,
    'patience': 12,
    'grad_clip_norm': 1.0,
    'dropout': 0.1},
   {'run_name': '{family}_joint_lowreg_5e4_wd5e4',
    'lr': 0.0005

## Run Transformer tuning


In [9]:
RUN_GRID = True
GRID_SEED = 42

all_results = []

if RUN_GRID:
    for family in ["standard", "set"]:
        for task_name, trials in TRANSFORMER_GRIDS.items():
            y_train, y_val = task_data(task_name)
            print(f"\n===== {family.upper()} Transformer tuning: {task_name} =====")
            for raw_spec in trials:
                spec = dict(raw_spec)
                spec["run_name"] = spec["run_name"].format(family=family)
                dropout = float(spec.pop("dropout"))
                print(f"\n--- {spec['run_name']} ---")

                set_seed(GRID_SEED)
                model = build_model(family, task_name, dropout=dropout)
                cfg = make_cfg(**spec)
                row = run_trial(model, y_train, y_val, cfg, family=family, task=task_name, seed=GRID_SEED)
                row["dropout"] = dropout
                row["params"] = count_trainable_params(model)

                if family == "standard":
                    row.update(STANDARD_ARCH)
                else:
                    row.update(SET_ARCH)

                all_results.append(row)

transformer_tuning_df = pd.DataFrame(all_results)
transformer_tuning_df



===== STANDARD Transformer tuning: joint =====

--- standard_joint_base_2e3_wd1e4 ---
epoch=001 train_loss=2.4847 val_loss=2.1345 score=0.1278
epoch=005 train_loss=0.5267 val_loss=0.4715 score=0.8074
epoch=010 train_loss=0.3234 val_loss=0.4358 score=0.8686

--- standard_joint_stable_1e3_wd1e4 ---
epoch=001 train_loss=2.4429 val_loss=2.4289 score=0.1962
epoch=005 train_loss=0.6806 val_loss=0.8275 score=0.7111
epoch=010 train_loss=0.2854 val_loss=0.3170 score=0.9181
epoch=015 train_loss=0.1224 val_loss=0.4099 score=0.9001
epoch=020 train_loss=0.0837 val_loss=0.4482 score=0.9001
epoch=025 train_loss=0.0583 val_loss=0.2579 score=0.9487
epoch=030 train_loss=0.0296 val_loss=0.2963 score=0.9505

--- standard_joint_reg_1e3_wd5e4 ---
epoch=001 train_loss=2.4362 val_loss=1.8737 score=0.2610
epoch=005 train_loss=0.6559 val_loss=0.7568 score=0.6751
epoch=010 train_loss=0.3997 val_loss=0.6895 score=0.7075
epoch=015 train_loss=0.1520 val_loss=0.2684 score=0.9397
epoch=020 train_loss=0.1211 val_loss

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,dropout,params,d_model,nhead,num_layers,dim_feedforward,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m,num_heads,num_sab_layers,num_seed_vectors
0,standard,joint,standard_joint_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,1.0,7,7.0,0.384609,0.348302,0.915392,0.915392,0.997300,0.917192,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,standard,joint,standard_joint_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,1.0,23,23.0,0.076801,0.282502,0.950495,0.950495,1.000000,0.950495,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,standard,joint,standard_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,1.0,29,29.0,0.035338,0.304360,0.950495,0.950495,1.000000,0.950495,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,standard,joint,standard_joint_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,1.0,26,26.0,0.044039,0.291321,0.942394,0.942394,0.999100,0.943294,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,standard,joint,standard_joint_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,25,25.0,0.054921,0.263657,0.946895,0.946895,0.999100,0.946895,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,standard,multitask,standard_mt_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,1.0,14,14.0,0.059586,0.247015,0.955896,0.955896,0.994599,0.956796,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,standard,multitask,standard_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,1.0,30,30.0,0.053569,0.227362,0.957696,0.957696,0.999100,0.957696,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,standard,multitask,standard_mt_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,1.0,28,28.0,0.064521,0.244919,0.954095,0.954095,0.999100,0.954095,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,standard,multitask,standard_mt_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,1.0,22,22.0,0.073655,0.240674,0.944194,0.944194,0.995500,0.946895,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,standard,multitask,standard_mt_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,27,27.0,0.056854,0.226731,0.952295,0.952295,0.996400,0.955896,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Select best candidate per family/task


In [10]:
summary_cols = [
    "family", "task", "run_name", "seed", "best_epoch", "score",
    "joint_accuracy", "building_accuracy", "floor_accuracy",
    "coordinate_mean_euclidean_m", "coordinate_rmse_m",
    "lr", "weight_decay", "dropout", "max_epochs", "patience", "grad_clip_norm", "params",
    "d_model", "nhead", "num_layers", "num_heads", "num_sab_layers", "dim_feedforward", "num_seed_vectors"
]

display(transformer_tuning_df[[c for c in summary_cols if c in transformer_tuning_df.columns]].sort_values(["family", "task", "score"], ascending=[True, True, False]))

best_by_family_task = (
    transformer_tuning_df
    .sort_values(["family", "task", "score"], ascending=[True, True, False])
    .groupby(["family", "task"], as_index=False)
    .first()
)

best_by_family_task


,family,task,run_name,seed,best_epoch,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean_m,coordinate_rmse_m,lr,weight_decay,dropout,max_epochs,patience,grad_clip_norm,params,d_model,nhead,num_layers,num_heads,num_sab_layers,dim_feedforward,num_seed_vectors
27,set,coordinate,set_coord_reg_1e3_wd5e4,42,37,-10.919728,NaN,NaN,NaN,10.919728,14.040860,0.0010,0.0005,0.1,50,10,1.0,481794,128,NaN,NaN,4.0,2.0,256,1.0
26,set,coordinate,set_coord_stable_1e3_wd1e4,42,37,-10.921133,NaN,NaN,NaN,10.921133,14.142971,0.0010,0.0001,0.1,50,10,1.0,481794,128,NaN,NaN,4.0,2.0,256,1.0
25,set,coordinate,set_coord_base_2e3_wd1e4,42,22,-11.658532,NaN,NaN,NaN,11.658532,15.524725,0.0020,0.0001,0.1,30,5,1.0,481794,128,NaN,NaN,4.0,2.0,256,1.0
28,set,coordinate,set_coord_low_5e4_wd1e4,42,39,-11.804937,NaN,NaN,NaN,11.804937,14.679612,0.0005,0.0001,0.1,60,12,1.0,481794,128,NaN,NaN,4.0,2.0,256,1.0
29,set,coordinate,set_coord_lowreg_5e4_wd5e4,42,39,-12.127828,NaN,NaN,NaN,12.127828,14.925200,0.0005,0.0005,0.1,80,15,1.0,481794,128,NaN,NaN,4.0,2.0,256,1.0
19,set,joint,set_joint_lowreg_5e4_wd5e4,42,43,0.944194,0.944194,0.999100,0.944194,NaN,NaN,0.0005,0.0005,0.1,80,15,1.0,483213,128,NaN,NaN,4.0,2.0,256,1.0
18,set,joint,set_joint_low_5e4_wd1e4,42,20,0.938794,0.938794,0.999100,0.938794,NaN,NaN,0.0005,0.0001,0.1,60,12,1.0,483213,128,NaN,NaN,4.0,2.0,256,1.0
17,set,joint,set_joint_reg_1e3_wd5e4,42,20,0.927993,0.927993,0.999100,0.927993,NaN,NaN,0.0010,0.0005,0.1,50,10,1.0,483213,128,NaN,NaN,4.0,2.0,256,1.0
16,set,joint,set_joint_stable_1e3_wd1e4,42,24,0.924392,0.924392,0.996400,0.924392,NaN,NaN,0.0010,0.0001,0.1,50,10,1.0,483213,128,NaN,NaN,4.0,2.0,256,1.0
15,set,joint,set_joint_base_2e3_wd1e4,42,1,0.036004,0.036004,0.241224,0.154815,NaN,NaN,0.0020,0.0001,0.1,30,5,1.0,483213,128,NaN,NaN,4.0,2.0,256,1.0


,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,dropout,params,d_model,nhead,num_layers,dim_feedforward,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m,num_heads,num_sab_layers,num_seed_vectors
0,set,coordinate,set_coord_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,1.0,37,37.0,0.010662,0.013415,-10.919728,NaN,NaN,NaN,0.1,481794,128,NaN,NaN,256,0.123422,0.162111,10.919728,14.04086,4.0,2.0,1.0
1,set,joint,set_joint_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,43,43.0,0.034788,0.337773,0.944194,0.944194,0.9991,0.944194,0.1,483213,128,NaN,NaN,256,NaN,NaN,NaN,NaN,4.0,2.0,1.0
2,set,multitask,set_mt_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,38,38.0,0.030049,0.228611,0.955896,0.955896,0.9982,0.956796,0.1,499080,128,NaN,NaN,256,NaN,NaN,NaN,NaN,4.0,2.0,1.0
3,standard,coordinate,standard_coord_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,1.0,48,48.0,0.010434,0.012527,-9.980649,NaN,NaN,NaN,0.1,349442,128,4.0,2.0,256,0.114569,0.154306,9.980649,13.16565,NaN,NaN,NaN
4,standard,joint,standard_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,1.0,29,29.0,0.035338,0.304360,0.950495,0.950495,1.0000,0.950495,0.1,350861,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,standard,multitask,standard_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,1.0,30,30.0,0.053569,0.227362,0.957696,0.957696,0.9991,0.957696,0.1,366728,128,4.0,2.0,256,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3-seed confirmation for selected Transformer configs


In [11]:
RUN_CONFIRM_FINAL = True
SEEDS_CONFIRM = (42, 123, 999)

confirm_rows = []

if RUN_CONFIRM_FINAL:
    for _, best in best_by_family_task.iterrows():
        family = best["family"]
        task_name = best["task"]
        y_train, y_val = task_data(task_name)

        spec = {
            "run_name": f"confirm_{best['run_name']}",
            "lr": float(best["lr"]),
            "weight_decay": float(best["weight_decay"]),
            "max_epochs": int(best["max_epochs"]),
            "patience": int(best["patience"]),
            "print_every": 5,
            "grad_clip_norm": None if pd.isna(best.get("grad_clip_norm", None)) else float(best["grad_clip_norm"]),
        }
        dropout = float(best["dropout"])

        for seed in SEEDS_CONFIRM:
            print(f"\n===== Confirm {family.upper()} Transformer {task_name}, seed={seed} =====")
            set_seed(seed)
            model = build_model(family, task_name, dropout=dropout)
            cfg = make_cfg(**spec)
            row = run_trial(model, y_train, y_val, cfg, family=family, task=task_name, seed=seed)
            row["selected_from_run"] = best["run_name"]
            row["dropout"] = dropout
            row["params"] = count_trainable_params(model)

            if family == "standard":
                row.update(STANDARD_ARCH)
            else:
                row.update(SET_ARCH)

            confirm_rows.append(row)

transformer_confirm_runs_df = pd.DataFrame(confirm_rows)
transformer_confirm_runs_df



===== Confirm SET Transformer coordinate, seed=42 =====
epoch=001 train_loss=0.6240 val_loss=0.2178 score=-50.9434
epoch=005 train_loss=0.0665 val_loss=0.0654 score=-29.6377
epoch=010 train_loss=0.0194 val_loss=0.0187 score=-14.3379
epoch=015 train_loss=0.0149 val_loss=0.0170 score=-13.9878
epoch=020 train_loss=0.0132 val_loss=0.0151 score=-12.1393
epoch=025 train_loss=0.0117 val_loss=0.0150 score=-12.1671
epoch=030 train_loss=0.0110 val_loss=0.0155 score=-12.3121
epoch=035 train_loss=0.0107 val_loss=0.0149 score=-12.2240
epoch=040 train_loss=0.0104 val_loss=0.0143 score=-11.4285
epoch=045 train_loss=0.0100 val_loss=0.0142 score=-11.5515

===== Confirm SET Transformer coordinate, seed=123 =====
epoch=001 train_loss=1.0159 val_loss=1.2596 score=-142.6338
epoch=005 train_loss=0.0579 val_loss=0.0517 score=-24.5665
epoch=010 train_loss=0.0291 val_loss=0.0310 score=-18.1998
epoch=015 train_loss=0.0209 val_loss=0.0252 score=-16.8225
epoch=020 train_loss=0.0148 val_loss=0.0167 score=-13.4469

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m,selected_from_run,dropout,params,d_model,num_heads,num_sab_layers,dim_feedforward,num_seed_vectors,joint_accuracy,building_accuracy,floor_accuracy,nhead,num_layers
0,set,coordinate,confirm_set_coord_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,1.0,37,37.0,0.010662,0.013415,-10.919728,0.123422,0.162111,10.919728,14.040860,set_coord_reg_1e3_wd5e4,0.1,481794,128,4.0,2.0,256,1.0,NaN,NaN,NaN,NaN,NaN
1,set,coordinate,confirm_set_coord_reg_1e3_wd5e4,123,0.0010,0.0005,256,512,50,10,5,1.0,31,31.0,0.012176,0.015095,-11.927882,0.132098,0.171883,11.927882,15.296345,set_coord_reg_1e3_wd5e4,0.1,481794,128,4.0,2.0,256,1.0,NaN,NaN,NaN,NaN,NaN
2,set,coordinate,confirm_set_coord_reg_1e3_wd5e4,999,0.0010,0.0005,256,512,50,10,5,1.0,35,35.0,0.011053,0.013754,-11.701690,0.126201,0.165179,11.701690,15.537414,set_coord_reg_1e3_wd5e4,0.1,481794,128,4.0,2.0,256,1.0,NaN,NaN,NaN,NaN,NaN
3,set,joint,confirm_set_joint_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,43,43.0,0.034788,0.337773,0.944194,NaN,NaN,NaN,NaN,set_joint_lowreg_5e4_wd5e4,0.1,483213,128,4.0,2.0,256,1.0,0.944194,0.9991,0.944194,NaN,NaN
4,set,joint,confirm_set_joint_lowreg_5e4_wd5e4,123,0.0005,0.0005,256,512,80,15,5,1.0,24,24.0,0.063250,0.299119,0.938794,NaN,NaN,NaN,NaN,set_joint_lowreg_5e4_wd5e4,0.1,483213,128,4.0,2.0,256,1.0,0.938794,0.9991,0.938794,NaN,NaN
5,set,joint,confirm_set_joint_lowreg_5e4_wd5e4,999,0.0005,0.0005,256,512,80,15,5,1.0,56,56.0,0.017075,0.364211,0.945995,NaN,NaN,NaN,NaN,set_joint_lowreg_5e4_wd5e4,0.1,483213,128,4.0,2.0,256,1.0,0.945995,0.9964,0.945995,NaN,NaN
6,set,multitask,confirm_set_mt_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,1.0,38,38.0,0.030049,0.228611,0.955896,NaN,NaN,NaN,NaN,set_mt_lowreg_5e4_wd5e4,0.1,499080,128,4.0,2.0,256,1.0,0.955896,0.9982,0.956796,NaN,NaN
7,set,multitask,confirm_set_mt_lowreg_5e4_wd5e4,123,0.0005,0.0005,256,512,80,15,5,1.0,41,41.0,0.017943,0.289587,0.951395,NaN,NaN,NaN,NaN,set_mt_lowreg_5e4_wd5e4,0.1,499080,128,4.0,2.0,256,1.0,0.951395,1.0000,0.951395,NaN,NaN
8,set,multitask,confirm_set_mt_lowreg_5e4_wd5e4,999,0.0005,0.0005,256,512,80,15,5,1.0,40,40.0,0.029297,0.320847,0.942394,NaN,NaN,NaN,NaN,set_mt_lowreg_5e4_wd5e4,0.1,499080,128,4.0,2.0,256,1.0,0.942394,0.9964,0.944194,NaN,NaN
9,standard,coordinate,confirm_standard_coord_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,1.0,48,48.0,0.010421,0.012643,-10.035570,0.115134,0.154969,10.035570,13.249066,standard_coord_stable_1e3_wd1e4,0.1,349442,128,NaN,NaN,256,NaN,NaN,NaN,NaN,4.0,2.0


## Confirmation summary


In [12]:
def summarize_confirm(df):
    if df.empty:
        return pd.DataFrame()
    agg = {
        "seed": "count",
        "score": ["mean", "std"],
        "best_epoch": "mean",
        "train_loss": "mean",
        "val_loss": "mean",
        "joint_accuracy": "mean",
        "building_accuracy": "mean",
        "floor_accuracy": "mean",
        "coordinate_mean_euclidean_m": "mean",
        "coordinate_rmse_m": "mean",
        "params": "first",
        "dropout": "first",
        "d_model": "first",
        "nhead": "first",
        "num_layers": "first",
        "num_heads": "first",
        "num_sab_layers": "first",
        "dim_feedforward": "first",
        "num_seed_vectors": "first",
    }
    available_agg = {k: v for k, v in agg.items() if k in df.columns}
    out = df.groupby(["family", "task", "run_name"]).agg(available_agg)
    out.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in out.columns]
    out = out.reset_index().rename(columns={"seed_count": "runs"})
    return out

transformer_confirmation_summary = summarize_confirm(transformer_confirm_runs_df)
transformer_confirmation_summary


,family,task,run_name,runs,score_mean,score_std,best_epoch_mean,train_loss_mean,val_loss_mean,joint_accuracy_mean,building_accuracy_mean,floor_accuracy_mean,coordinate_mean_euclidean_m_mean,coordinate_rmse_m_mean,params_first,dropout_first,d_model_first,nhead_first,num_layers_first,num_heads_first,num_sab_layers_first,dim_feedforward_first,num_seed_vectors_first
0,set,coordinate,confirm_set_coord_reg_1e3_wd5e4,3,-11.516433,0.528993,34.333333,0.011297,0.014088,NaN,NaN,NaN,11.516433,14.958207,481794,0.1,128,NaN,NaN,4.0,2.0,256,1.0
1,set,joint,confirm_set_joint_lowreg_5e4_wd5e4,3,0.942994,0.003747,41.000000,0.038371,0.333701,0.942994,0.9982,0.942994,NaN,NaN,483213,0.1,128,NaN,NaN,4.0,2.0,256,1.0
2,set,multitask,confirm_set_mt_lowreg_5e4_wd5e4,3,0.949895,0.006875,39.666667,0.025763,0.279682,0.949895,0.9982,0.950795,NaN,NaN,499080,0.1,128,NaN,NaN,4.0,2.0,256,1.0
3,standard,coordinate,confirm_standard_coord_stable_1e3_wd1e4,3,-10.299660,0.407666,41.666667,0.010851,0.013173,NaN,NaN,NaN,10.299660,13.511245,349442,0.1,128,4.0,2.0,NaN,NaN,256,NaN
4,standard,joint,confirm_standard_joint_reg_1e3_wd5e4,3,0.955896,0.007368,37.333333,0.027741,0.291244,0.955896,0.9979,0.956196,NaN,NaN,350861,0.1,128,4.0,2.0,NaN,NaN,256,NaN
5,standard,multitask,confirm_standard_mt_stable_1e3_wd1e4,3,0.958596,0.003118,32.000000,0.048611,0.233516,0.958596,0.9988,0.959496,NaN,NaN,366728,0.1,128,4.0,2.0,NaN,NaN,256,NaN


## Final config dictionary to copy into baseline/robustness notebooks


In [13]:
FINAL_TUNED_CFGS = {
    family: {
        row["task"]: {
            "lr": float(row["lr"]),
            "weight_decay": float(row["weight_decay"]),
            "dropout": float(row["dropout"]),
            "grad_clip_norm": None if pd.isna(row.get("grad_clip_norm", None)) else float(row["grad_clip_norm"]),
            "max_epochs": int(row["max_epochs"]),
            "patience": int(row["patience"]),
            "print_every": 5,
            "batch_size": 256,
            "val_batch_size": 512,
            "architecture": (
                {k: int(row[k]) for k in ["d_model", "nhead", "num_layers", "dim_feedforward"] if k in row and pd.notna(row[k])}
                if family == "standard"
                else {k: int(row[k]) for k in ["d_model", "num_heads", "num_sab_layers", "dim_feedforward", "num_seed_vectors"] if k in row and pd.notna(row[k])}
            ),
        }
        for _, row in g.iterrows()
    }
    for family, g in best_by_family_task.groupby("family")
}

FINAL_TUNED_CFGS


{'set': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'dim_feedforward': 256,
    'num_seed_vectors': 1}},
  'joint': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'dim_feedforward': 256,
    'num_seed_vectors': 1}},
  'multitask': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'di

## Save outputs


In [14]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not transformer_tuning_df.empty:
    transformer_tuning_df.to_csv(OUTPUT_DIR / "transformer_tuning_grid_results.csv", index=False)
if not transformer_confirm_runs_df.empty:
    transformer_confirm_runs_df.to_csv(OUTPUT_DIR / "transformer_confirm_runs.csv", index=False)
if not transformer_confirmation_summary.empty:
    transformer_confirmation_summary.to_csv(OUTPUT_DIR / "transformer_confirmation_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)


Saved outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_tuning
